# Getting started with APGAP

Hi. This is one of five notebooks pre-loaded into your APGAP project's
Vertex AI Workbench. Open this one first.

It's written for people new to the platform. You'll want to be
comfortable enough with Python to read a few cells, but no
pathogen-genomics background is assumed.

What it does:

- Confirms the kernel is up and your project is configured
- Lists what data your project has access to
- Previews one file so you can see it's real
- Points you at the other four notebooks

It doesn't install anything, launch any pipelines, or modify data.
Safe to re-run whenever. About two minutes top to bottom.

> **Kernel:** When JupyterLab prompts "Select Kernel," pick **`Python 3 (ipykernel) (Local)`** (the first option under "Start python Kernel"). The PyTorch and TensorFlow kernels are missing libraries this notebook needs — choosing them will cause early cells to fail with `ModuleNotFoundError`. If you already see `Python 3 (ipykernel) (Local)` in the top-right of this tab, you're good to go.

In [ ]:
import subprocess

DETECTED_PROJECT = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True
).strip()

# Defaults are auto-detected from the Workbench environment.
# Override these only if you want to point at something different.
PROJECT_ID  = DETECTED_PROJECT                # @param {type:"string"}
REGION      = "us-central1"                   # @param {type:"string"}
DATASET_URI = ""                              # @param {type:"string"}
# Leave DATASET_URI blank to auto-pick the first *-analytical-dataset-* bucket.

## Where am I?

Quick sanity check. Your Workbench is scoped to one APGAP project.
Its GCP project ID and the service account it runs as are below.

In [ ]:
import subprocess

active_account = subprocess.check_output(
    ["gcloud", "config", "get-value", "account"], text=True
).strip()

print(f"GCP project: {PROJECT_ID}")
print(f"Region:      {REGION}")
print(f"Identity:    {active_account}")

## What's in my project?

Each project has two kinds of Cloud Storage bucket.

**Output buckets** (`gs://<pathogen>-seqera-output-*`) are where
pipeline runs write their results. Empty until you've run something.

**Analytical-dataset buckets** (`gs://<pathogen>-analytical-dataset-*`)
hold input files, one bucket per dataset shown in your project's
Datasets tab on the APGAP web app. The "analytical" naming is a little
misleading: these are raw inputs (FASTQs and so on), not curated
results.

Here's what your Workbench can see.

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
buckets = fs.ls("")
print(f"Visible buckets ({len(buckets)}):")
for b in buckets:
    role = (
        "Output Bucket"     if "seqera-output"      in b else
        "Analytical Dataset" if "analytical-dataset" in b else
        "Other"
    )
    print(f"  gs://{b:<60s}  [{role}]")

## Preview your data

The next cell lists files in your dataset. It defaults to the first
analytical-dataset bucket it sees, preferring one whose name starts
with your project's pathogen prefix. To use a different dataset, copy
the URI from the Datasets tab and paste it into `DATASET_URI` above.

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
if not DATASET_URI:
    all_analytical = [b.rstrip("/") for b in fs.ls("") if "analytical-dataset" in b]
    # Prefer buckets whose name starts with this project's pathogen prefix
    # (the leading token of PROJECT_ID, e.g. "h1n1" from "h1n1-apgap-project-0a47").
    pathogen = PROJECT_ID.split("-")[0]
    matched = [b for b in all_analytical if b.startswith(pathogen)]
    candidates = matched or all_analytical
    if not candidates:
        print("⚠ No analytical-dataset buckets visible. Set DATASET_URI manually.")
        DATASET_URI = ""
    else:
        DATASET_URI = f"gs://{candidates[0]}/"
        if len(all_analytical) > 1:
            others = [b for b in all_analytical if b != candidates[0]]
            print(f"Auto-selected dataset: {DATASET_URI}")
            print(f"  (other analytical-dataset buckets visible: {others})\n")
        else:
            print(f"Auto-selected dataset: {DATASET_URI}\n")

def _fmt_size(n_bytes):
    if n_bytes < 1024:                return f"{n_bytes:>6} B "
    if n_bytes < 1024 * 1024:         return f"{n_bytes/1024:>6.1f} KB"
    return f"{n_bytes/1024/1024:>6.1f} MB"

if DATASET_URI:
    all_files = fs.ls(DATASET_URI.replace("gs://", "").rstrip("/"))
    # Files starting with `_` are platform internals (e.g. metadata sidecars).
    # Hidden from the listing by default; users see the count.
    data_files   = [f for f in all_files if not f.rsplit("/", 1)[-1].startswith("_")]
    hidden_files = [f for f in all_files if     f.rsplit("/", 1)[-1].startswith("_")]

    print(f"Files in this dataset ({len(data_files)} data file(s)):")
    for f in data_files:
        size = _fmt_size(fs.info(f).get("size", 0))
        print(f"  {size}   gs://{f}")
    if hidden_files:
        print(f"\n(plus {len(hidden_files)} platform metadata file(s), hidden by default)")

## What you can do here

Four companion notebooks live alongside this one. The first three are
task-focused; the fourth is a reference you'll dip into when a file
format or output layout question comes up.

| Task or reference | Notebook |
|---|---|
| Read your data into Python, peek at FASTQ records, get basic stats | [02-read-your-data.ipynb](02-read-your-data.ipynb) |
| Install Nextflow and launch a pipeline on GCP Batch (using `nf-core/viralrecon`) | [03-launch-a-pipeline.ipynb](03-launch-a-pipeline.ipynb) |
| Pull additional sequences from NCBI SRA, ENA, or GenBank | [04-download-from-public-repos.ipynb](04-download-from-public-repos.ipynb) |
| File format reference: FASTQ, VCF, samplesheet, viralrecon output layout | [05-reference.ipynb](05-reference.ipynb) |

Open whichever you need. The task notebooks run standalone; you don't
have to have touched the others first. The reference notebook is for
skimming or searching, not running.

## Stuck?

- Ask your project lead
- Vertex AI Workbench docs: https://cloud.google.com/vertex-ai/docs/workbench/introduction
- nf-core pipeline catalog: https://nf-co.re